IMPORTANT

ROIs should be saved as Staining Method_Genotype_Gender_Brain_Slide_Region (e.g., ThioS_hTauNLF_M_1116097_S1_HIP).

Make sure to set the right ilastik project directories for your computer.

Whenever ilastik is called, change project directory to whichever project you need for the brains you have (i.e., if you are quantifying NLGF brains, change 'NLF_Segmentation.ilp' to 'NLGF_Segmentation.ilp').

In [ ]:
from pathlib import Path
import subprocess

#Folder containing ROIs and folder for Ilastik outputs.
in_folder = Path("E:\\260306_AxonaThioS_Tiffs\\Headless_Test_In")
out_folder = Path("E:\\260306_AxonaThioS_Tiffs\\Headless_Test_Out")

#Ilastik segmentation.
for in_file in in_folder.glob("*.tif"):
    out_file = out_folder / in_file.name.replace(".tif", "_Probabilities.h5")
    print(f"Processing {in_file} -> {out_file}")
    command = [
    "C:\\Program Files\\ilastik-1.4.2b6\\ilastik.exe",
    '--headless',
    '--project=E:\\260323_NLGF_Axona_Pixel_Classification_Probabilites_OverlabelledPlaques',
    '--output_format=hdf5',
    '--export_source=Probabilities',
    '--raw_data=' + str(in_file),
    '--output_filename_format=' + str(out_file)
]


    print("COMMAND=", "  ".join(command))

    process = subprocess.run(command, check = True)

In [ ]:
import h5py
import pandas as pd
import numpy as np
from pathlib import Path

# Define the output folder and glob for probability files
out_folder = Path("E:\\260306_AxonaThioS_Tiffs\\Headless_Test_Out")  # Replace with your actual folder path
h5_files = out_folder.glob("*_Probabilities.h5")

# Dictionary to store results
h5_dict = {}

# Choose which class you want to measure coverage for
# NOTE: Python is 0-indexed, so last class (previously 3) is index 2
target_class = 2  # Plaques class, change this index if necessary
threshold = 0.5  # Threshold for converting probabilities to hard labels

for in_file in h5_files:
    filename = in_file.stem
    parts = filename.split("_")
    if len(parts) >= 6:
        staining_method = parts[0]
        genotype = parts[1]
        gender = parts[2]
        brain = parts[3]
        slide = parts[4]
        region = parts[5]
        print(f"Processing: {filename}")

        # Load HDF5 probabilities
        with h5py.File(in_file, 'r') as f:
            if "Probabilities" in f.keys():
                dataset_name = "Probabilities"
            else:
                dataset_name = list(f.keys())[0]  # Select the first dataset if Probabilities isn't available
            data = f[dataset_name][:]  # shape (H, W, C), H: height, W: width, C: number of classes

        # Apply threshold to convert soft labels to hard labels for the target class (plaque class)
        plaque_data = data[..., target_class] >= threshold  # Boolean mask: True where probability > threshold
        percentage_coverage = np.mean(plaque_data) * 100  # Percentage of pixels considered plaques

        # Store results in dictionary
        unique_key = f"{staining_method}_{genotype}_{gender}_{brain}_{region}"
        if unique_key not in h5_dict:
            h5_dict[unique_key] = {'slide': [], 'a': []}
        h5_dict[unique_key]['slide'].append(slide)
        h5_dict[unique_key]['a'].append(percentage_coverage)

# Write results to Excel
for key, value in h5_dict.items():
    slide = value['slide']
    a = value['a']
    # Compute average percentage coverage across slides
    total_percentage_coverage = sum(a) / len(a)
    a.append(total_percentage_coverage)
    slide.append("AVERAGE")

    # Replace "*" with "_" in the key for the output file
    b = key.replace("*", "_")

    # Create DataFrame for output
    e = zip(slide, a)
    df = pd.DataFrame(e, columns=["Slide", "Percentage Coverage"])

    # Define output filename and save to Excel
    output_filename = f"{b}_Percentage_Of_Coverage.xlsx"
    output_path = out_folder / output_filename

    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Percentage of Coverage', index=False)
    print(f"Saved Excel file: {output_filename}")

In [ ]:
#Ilastik plaque classification
for in_file in in_folder.glob("*.tif"):
    out_file_1 = out_folder / in_file.name.replace(".tif", "_Object_Predictions.h5")
    out_file_2 = out_folder / in_file.name.replace(".tif", "_Table.csv")
    pred = out_folder / in_file.name.replace(".tif", "_Probabilities.h5")
    print(f"Processing {in_file} -> {out_file}")
    command = [
    "C:\\ProgramData\\Microsoft\\Windows\\Start Menu\\Programs\\ilastik-1.4.2b6\\ilastik.lnk",
    '--headless',
    '--project=E:\\260306_AxonaThioS_Tiffs\\NLGF_Plaque_Classification.ilp',
    '--table_filename=' + str(out_file_2),
    '--output_format=hdf5',
    '--export_source=Object Predictions',
    '--raw_data=' + str(in_file),
    '--prediction_maps=' + str(pred),
    '--output_filename_format=' + str(out_file)
]


    print("COMMAND=", "  ".join(command))

    process = subprocess.run(command, check = True)

In [ ]:
#Quantification of plaque classification
csv_files = out_folder.glob("*.csv")

csv_dict = {}

for in_file in csv_files:
    filename = in_file.stem
    parts = filename.split("_")

    if len(parts) >= 6:
        staining_method = parts[0]
        genotype = parts[1]
        gender = parts[2]
        brain = parts[3]
        slide = parts[4]
        region = parts[5]

        unique_key = f"{brain}_{region}"

        if unique_key not in csv_dict:
            csv_dict[unique_key] = {'data': []}

        read = pd.read_csv(in_file)
        csv_dict[unique_key]['data'].append(read)

for key, value in csv_dict.items():
    data = pd.concat(value['data'])
    c = data['Predicted Class'].value_counts(normalize=True) * 100

    class_dist_row = pd.DataFrame([c], columns=c.index)
    result = pd.concat([class_dist_row, data], ignore_index=True)

    output_filename = f"{key}_SML.xlsx"
    output_path = out_folder / output_filename

    writer = pd.ExcelWriter(output_path, engine='xlsxwriter')
    result.to_excel(writer, sheet_name='SML', index=False)
    writer.close()